# Pancancer enrichment analysis step 2: Find enriched pathways using Reactome's Analysis Service API for enrichment analysis

## Setup

In [1]:
import cptac
import cptac.utils as ut
import pandas as pd
import numpy as np
import os
import datetime

In [2]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

STEP1_DIR = "step1_outputs"
STEP1_FILE = "tumor_change_20200529_195104_10000_perm.tsv"
STEP1_FILE_PATH = os.path.join(STEP1_DIR, STEP1_FILE)

STEP2_DIR = "step2_outputs"
STEP2_FILE = f"enrichment_reactome_{TIME_START}_from_{STEP1_FILE.rsplit('.', maxsplit=1)[0]}.tsv"
STEP2_FILE_PATH = os.path.join(STEP2_DIR, STEP2_FILE)

if not os.path.isdir(STEP2_DIR):
    os.mkdir(STEP2_DIR)

## Prepare data

In [3]:
# Read in the file from step 1
data = pd.read_csv(STEP1_FILE_PATH, sep='\t', index_col=0)

# The ranked enrichment analysis service wants ranking scores
# where "bigger is better", such as expression values or
# t scores. We are ranking by adjusted p-values, where smaller
# is better. So, we'll create a column of (1 - adj_p) to use
# for ranking.
data = data.assign(one_minus_adj_p=1 - data["adj_p"])

# Make a column where all increases are +1 and all decreases 
# are -1, because these are ratioed abundances, so we can't 
# compare magnitudes between genes--we can only compare whether 
# there was a change, and whether it was positive or negative
data = data.assign(simplified_change=data["change"])

data.loc[data["change"] > 0, "simplified_change"] = 1
data.loc[data["change"] < 0, "simplified_change"] = -1
data.loc[data["change"] == 0, "simplified_change"] = 0

In [4]:
data.head()

,cancer_type,protein,change,P_val,adj_p,reject_null,protein_str,one_minus_adj_p,simplified_change
0,ccrcc,"('A1BG', 'NP_570602.2')",0.282928,0.0000,0.00000,True,A1BG,1.00000,1.0
1,ccrcc,"('A1CF', 'NP_620310.1')",-0.551358,0.0000,0.00000,True,A1CF,1.00000,-1.0
2,ccrcc,"('A2M', 'NP_000005.2')",0.595512,0.0000,0.00000,True,A2M,1.00000,1.0
3,ccrcc,"('A4GALT', 'NP_001304967.1')",0.479410,0.1735,0.21431,False,A4GALT,0.78569,1.0
4,ccrcc,"('AAAS', 'NP_056480.1')",0.173579,0.0000,0.00000,True,AAAS,1.00000,1.0


## Run enrichment analysis

In [5]:
# For each cancer, find enriched pathways based on p value for differential expression   
all_enrichments = pd.DataFrame()

for cancer_type in data["cancer_type"].unique():
    
    ranked_data = data.loc[data["cancer_type"] == cancer_type, ["protein_str", "one_minus_adj_p"]].\
        set_index("protein_str").\
        rename(columns={"one_minus_adj_p": f"{cancer_type}_one_minus_adj_p"})

    unneeded_token, cancer_enriched = ut.reactome_enrichment_analysis(
        analysis_type="ranked", 
        data=ranked_data, 
        sort_by="ENTITIES_FDR", 
        ascending=True,
        include_high_level_diagrams=False, 
        include_interactors=False)
    
    # Record the cancer type and rename the pathway id column
    cancer_enriched = cancer_enriched.\
        assign(cancer_type=cancer_type).\
        rename(columns={"stId": "pathway_id"})
    
    # Append to the overall dataframe
    all_enrichments = all_enrichments.append(cancer_enriched)

In [6]:
# Drop columns we don't need
all_enrichments = all_enrichments.drop(columns=["entities_ratio", "entities_found", "entities_total"])

In [7]:
# Assign pathway ranks within each cancer type based on p value
# We use p value instead of FDR because they will both put the pathways
# in the same order, but the p values are more precise for ranking--some
# pathways that have the same FDR have different p values, so we get
# more specific ranks from the p values.

all_enrichments = all_enrichments.\
    assign(cancer_rank_pval=all_enrichments.groupby("cancer_type")["entities_pValue"].rank()).\
    sort_values(by=["cancer_type", "cancer_rank_pval"]).\
    reset_index(drop=True)

## Summarize enrichment results from all cancers

In [8]:
enrichment_summary = all_enrichments[["pathway_id", "name"]].drop_duplicates(keep="first")

pathway_times_enriched = enrichment_summary["pathway_id"].apply(
    lambda x: all_enrichments[all_enrichments["pathway_id"] == x].shape[0])

avg_rank = enrichment_summary["pathway_id"].apply(
    lambda x: all_enrichments.loc[all_enrichments["pathway_id"] == x, "cancer_rank_pval"].mean())

enrichment_summary = enrichment_summary.\
    assign(
        pathway_times_enriched=pathway_times_enriched,
        pathway_avg_rank=avg_rank).\
    sort_values(
        by=["pathway_times_enriched", "pathway_avg_rank"],
        ascending=[False, True]).\
    reset_index(drop=True)

In [9]:
enrichment_summary

,pathway_id,name,pathway_times_enriched,pathway_avg_rank
0,R-HSA-6798695,Neutrophil degranulation,8,1.000000
1,R-HSA-6791226,Major pathway of rRNA processing in the nucleo...,8,2.875000
2,R-HSA-72163,mRNA Splicing - Major Pathway,8,5.375000
3,R-HSA-72172,mRNA Splicing,8,10.500000
4,R-HSA-72689,Formation of a pool of free 40S subunits,8,12.750000
5,R-HSA-1236977,Endosomal/Vacuolar pathway,8,13.625000
6,R-HSA-927802,Nonsense-Mediated Decay (NMD),8,17.750000
7,R-HSA-975957,Nonsense Mediated Decay (NMD) enhanced by the ...,8,17.750000
8,R-HSA-72203,Processing of Capped Intron-Containing Pre-mRNA,8,18.500000
9,R-HSA-1799339,SRP-dependent cotranslational protein targetin...,8,20.500000


## Visualize pathways with expression change

In [10]:
# Submit the differential expression data for each cancer type to the analysis service, and get the tokens
# The data we'd submitted previously were the p values for the differential expression, but that's not what
# we want to visualize; we want to see whether the expression was up or down.
diff_expr_tokens = {}

for cancer_type in data["cancer_type"].unique():

    # Select data for that cancer type
    # Exclude samples where not reject_null
    omics = data.loc[(data["cancer_type"] == cancer_type) & data["reject_null"], 
                     ["protein_str", "simplified_change"]].\
        set_index("protein_str").\
        rename(columns={"simplified_change": f"{cancer_type}_simp_change"})

    analysis_token, unneeded_df = ut.reactome_enrichment_analysis(
        analysis_type="ranked", 
        data=omics, 
        sort_by="ENTITIES_FDR", 
        ascending=True,
        include_high_level_diagrams=True, 
        include_interactors=False)
    
    diff_expr_tokens[cancer_type] = analysis_token

In [11]:
# Select pathways to get URLs for
to_viz = enrichment_summary.iloc[0:10, :]

# Get URLs
id_list = []
cancer_type_list = []
expr_list = []
url_list = []

for idx in to_viz.index:
    pathway_id = to_viz.loc[idx, "pathway_id"]
    
    for cancer_type in data["cancer_type"].unique():
         
        # Get the URL
        expr_vals, url = ut.reactome_pathway_overlay(
            pathway=pathway_id,
            analysis_token=diff_expr_tokens[cancer_type],
            open_browser=False)
        
        id_list.append(pathway_id)
        cancer_type_list.append(cancer_type)
        expr_list.append(expr_vals[0])
        url_list.append(url)

In [12]:
# Put all the urls in a dataframe
urls_df = pd.DataFrame({
        "pathway_id": id_list,
        "cancer_type": cancer_type_list,
        "mean_expr": expr_list,
        "url": url_list})

# Join in the enrichment summary table
urls_df = urls_df.merge(
    enrichment_summary,
    how="left",
    left_on="pathway_id",
    right_on="pathway_id",
    validate="many_to_one")

# Join in the original enrichment data for each pathway and cancer type
urls_df = urls_df.merge(
    all_enrichments,
    how="left",
    left_on=["pathway_id", "name", "cancer_type"],
    right_on=["pathway_id", "name", "cancer_type"],
    validate="one_to_one")

In [13]:
# Print the dataframe in such a way that the links are clickable
urls_df.style.format('<a href="{0}">{0}</a>', subset="url")

,pathway_id,cancer_type,mean_expr,url,name,pathway_times_enriched,pathway_avg_rank,entities_pValue,entities_fdr,cancer_rank_pval
0,R-HSA-6798695,ccrcc,0.233503,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDkxNjQ1MDNfMjc1,Neutrophil degranulation,8,1.000000,0.003467,0.842640,1.000000
1,R-HSA-6798695,colon,-0.134831,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDkxNjQ1MDZfMjc2,Neutrophil degranulation,8,1.000000,0.000000,0.000000,1.000000
2,R-HSA-6798695,endometrial,0.288515,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDkxNjQ1MDlfMjc3,Neutrophil degranulation,8,1.000000,0.001403,0.821921,1.000000
3,R-HSA-6798695,gbm,0.166227,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDkxNjQ1MTNfMjc4,Neutrophil degranulation,8,1.000000,0.002130,0.803798,1.000000
4,R-HSA-6798695,hnscc,0.133159,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDkxNjQ1MTZfMjc5,Neutrophil degranulation,8,1.000000,0.003649,0.832921,1.000000
5,R-HSA-6798695,lscc,-0.407595,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDkxNjQ1MTlfMjgw,Neutrophil degranulation,8,1.000000,0.001638,0.872650,1.000000
6,R-HSA-6798695,luad,-0.410000,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDkxNjQ1MjNfMjgx,Neutrophil degranulation,8,1.000000,0.000180,0.441228,1.000000
7,R-HSA-6798695,ovarian,-0.303754,https://reactome.org/PathwayBrowser/#/R-HSA-6798695&DTAB=AN&ANALYSIS=MjAyMDA2MDkxNjQ1MjZfMjgy,Neutrophil degranulation,8,1.000000,0.000035,0.086606,1.000000
8,R-HSA-6791226,ccrcc,0.699422,https://reactome.org/PathwayBrowser/#/R-HSA-6791226&DTAB=AN&ANALYSIS=MjAyMDA2MDkxNjQ1MDNfMjc1,Major pathway of rRNA processing in the nucleolus and cytosol,8,2.875000,0.031321,0.842640,4.000000
9,R-HSA-6791226,colon,0.771429,https://reactome.org/PathwayBrowser/#/R-HSA-6791226&DTAB=AN&ANALYSIS=MjAyMDA2MDkxNjQ1MDZfMjc2,Major pathway of rRNA processing in the nucleolus and cytosol,8,2.875000,0.000000,0.000138,2.000000


In [14]:
# Save the results
urls_df.to_csv(STEP2_FILE_PATH, sep='\t')